In [51]:
import os
import pandas as pd
import numpy as np
import random
from glob import glob

In [21]:
def count_annotations_flag(filename, label, path):
    annotated_images = os.listdir(path)
    annotated_images.remove('.DS_Store')
    filename = filename.replace(".geojson","")
    if filename in annotated_images:
        if os.path.exists(os.path.join(path,filename, label)):
            items = os.listdir(os.path.join(path,filename, label))
            #items = [len(os.listdir(os.path.join(path,filename,i))) for i in wgm_dir]
            return len(items)
        else:
            print("folder does not exist")
            return 0
    else:
        return None

In [22]:
data_dir =  "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"
redmarked_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images_RedMarked/"

In [23]:
all_files = os.listdir(data_dir)
for f in os.listdir(redmarked_dir):
    all_files.remove(f)

In [24]:
annotations_df = pd.DataFrame({"filename": all_files,"filepath":[os.path.join(data_dir,x) for x in all_files]})

In [25]:
image_saved_path = "/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/WM_images"
annotations_df["WM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"White",image_saved_path))
annotations_df["GM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"grey",image_saved_path))
annotations_df["bg_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"bg",image_saved_path))

In [26]:
redmarked_list = os.listdir(redmarked_dir)
redmarked_list.remove('.DS_Store')
annotations_df1 = pd.DataFrame({"filename": redmarked_list,"filepath":[os.path.join(redmarked_dir,x) for x in redmarked_list]})

In [27]:
image_saved_path = "/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/WM_images_RedMarked"
annotations_df1["WM_count"]=annotations_df1["filename"].apply(lambda l:count_annotations_flag(l,"White",image_saved_path))
annotations_df1["GM_count"]=annotations_df1["filename"].apply(lambda l:count_annotations_flag(l,"grey",image_saved_path))
annotations_df1["bg_count"]=annotations_df1["filename"].apply(lambda l:count_annotations_flag(l,"bg",image_saved_path))

In [28]:
annotation_df = pd.concat([annotations_df,annotations_df1],ignore_index=True)

In [29]:
def map_tag(filename):
    if filename.startswith("PD"):
        return "PDD"
    else:
        return "DLB"
    
annotation_df["LBD_tag"]=annotation_df["filename"].apply(lambda l : map_tag(l))
annotation_df = annotation_df.sort_values(by="filename").reset_index()
annotation_df["index"]=annotation_df.index

In [30]:
annotation_df

,index,filename,filepath,WM_count,GM_count,bg_count,LBD_tag
0,0,11_063_CG_aSyn_x200.svs,/gladstone/finkbeiner/steve/work/data/npsad_da...,3678,4830,2112,DLB
1,1,12_007_CG_aSyn_x200.svs,/gladstone/finkbeiner/steve/work/data/npsad_da...,2204,1802,2891,DLB
2,2,12_060_CG_aSyn_x200.svs,/gladstone/finkbeiner/steve/work/data/npsad_da...,5144,6832,4512,DLB
3,3,13_131_CG_aSyn_x200.svs,/gladstone/finkbeiner/steve/work/data/npsad_da...,4878,5416,3024,DLB
4,4,13_177_CG_aSyn_x200.svs,/gladstone/finkbeiner/steve/work/data/npsad_da...,3059,3148,4844,DLB
5,5,14_036_CG_aSyn_x200.svs,/gladstone/finkbeiner/steve/work/data/npsad_da...,3903,3965,2728,DLB
6,6,14_053_CG_aSyn_x200.svs,/gladstone/finkbeiner/steve/work/data/npsad_da...,2189,3402,2293,DLB
7,7,14_073_CG_aSyn_x200.svs,/gladstone/finkbeiner/steve/work/data/npsad_da...,5924,6405,3960,DLB
8,8,14_075_CG_aSyn_x200.svs,/gladstone/finkbeiner/steve/work/data/npsad_da...,3840,4741,3084,DLB
9,9,14_087_CG_aSyn_x200.svs,/gladstone/finkbeiner/steve/work/data/npsad_da...,782,880,1758,DLB


In [40]:
random_list = random.sample(range(0, 31), 20)
annotation_df["train_test_flag"]=np.where(annotation_df["index"].isin(random_list),"Train","Val")
annotation_df.groupby(["train_test_flag","LBD_tag"])["filename"].count()

train_test_flag  LBD_tag
Train            DLB        10
                 PDD        10
Val              DLB         5
                 PDD         6
Name: filename, dtype: int64

In [42]:
annotation_df.groupby(["train_test_flag","LBD_tag"]).agg({"WM_count":"sum","GM_count":"sum","bg_count":"sum"})

WM_count  GM_count  bg_count
train_test_flag LBD_tag                              
Train           DLB         35487     37507     38549
                PDD         17394     24569     19283
Val             DLB         17280     20742     15508
                PDD          6352      8644      8265

In [46]:
annotation_df.to_csv("/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/Intermediate_data/WM_Annotations_redmarked.csv")

In [49]:
annotation_df["filepath"].iloc[0]

'/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/11_063_CG_aSyn_x200.svs'

In [52]:
def map_label(class_name):
    if class_name=="White":
        return 0
    if class_name=="grey":
        return 1
    if class_name=="bg":
        return 2
    return -1

def create_df_from_folders(train_test_flag):
    patient_df = annotation_df[annotation_df["train_test_flag"]==train_test_flag]
    #data_dir =  "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"
    df = pd.DataFrame(columns=["WSI_filename","crop_filepath","class"])
    ind = 0
    for index in patient_df.index:
        folder_name = annotation_df.iloc[index]["filepath"]
        for label_name in ["White","grey","bg"]:
            files = glob(os.path.join(folder_name, label_name, '*.png'))
            #print(len(files))
            temp_df =  pd.DataFrame({"crop_filepath":files})
            temp_df["WSI_filename"] = folder_name
            temp_df["class"] = label_name
            df = pd.concat([df, temp_df], ignore_index=True)
    df["label"]=df["class"].apply(lambda l:map_label(l))
    return df

train_df = create_df_from_folders("Train")
val_df = create_df_from_folders("Val")
#test_df = create_df_from_folders("Test")

print("Training Data Size :", len(train_df))
print("Validation Data Size :", len(val_df))
#print("Test Data Size : ", len(test_df))

Training Data Size : 172789
Validation Data Size : 76791


In [53]:
train_df

,WSI_filename,crop_filepath,class,label
0,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,White,0
1,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,White,0
2,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,White,0
3,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,White,0
4,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,White,0
...,...,...,...,...
172784,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,bg,2
172785,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,bg,2
172786,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,bg,2
172787,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,bg,2


In [54]:
val_df

,WSI_filename,crop_filepath,class,label
0,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,White,0
1,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,White,0
2,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,White,0
3,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,White,0
4,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,White,0
...,...,...,...,...
76786,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,bg,2
76787,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,bg,2
76788,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,bg,2
76789,/gladstone/finkbeiner/steve/work/data/npsad_da...,/gladstone/finkbeiner/steve/work/data/npsad_da...,bg,2


In [55]:
train_df.to_csv("/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/Intermediate_data/train_redmarked_full.csv")
val_df.to_csv("/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/Intermediate_data/val_redmarked_full.csv")

In [57]:
train_df[["WSI_filename","crop_filepath","label"]].to_csv("/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/Intermediate_data/train_redmarked.csv")
val_df[["WSI_filename","crop_filepath","label"]].to_csv("/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/Intermediate_data/val_redmarked.csv")